# Prompt Engineering 101 - Part II. HOMEWORK
## The Syntax of Semantics

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini API)
# We are connecting to Google's Gemini models.

# 1. Install the Google AI SDK
!pip install -q -U google-genai


from google.colab import userdata
import google.genai as genai

from IPython.display import display, Markdown


# 2. Configure the API Key
# Go to https://aistudio.google.com/app/apikey to get a key.
# It is free and takes 1 click.

# 3. Use Colab Secrets (Best Practice)
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 4. Create Wrapper Class for querying
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e


# Free tier models have limits. In case we run out of a model quota, we'll 
# use one of the fallback models.
#
# Possible model names we can use are:
# - gemini-2.5-flash-lite
# - gemini-2.5-flash
# - gemini-3-flash-preview
try:
    model = GeminiModel(GEMINI_API_KEY, model_name='gemini-2.5-flash')
    print("✅ Connection Established. The Engine is ready.")
except Exception as e:
    print(f"❌ Error: {e}. Did you paste your API key?")

# 🏠 Homework: The "Crisis Bot" Architect

### The Scenario
You are the Communications Director for **"CloudFly,"** a fictional airline.
A video has just gone viral showing a flight attendant being rude to a passenger's cat.
Twitter/X is exploding with angry messages.

### The Task
Build a **Python Prompt Template** (using the variables method we learned) that generates personalized apologies to angry customers.

### Requirements (Grading Rubric)
1.  **Persona:** The AI must adopt a "Empathetic, Accountable, but Professional" persona. It cannot sound robotic.
2.  **CO-STAR:** You must use the full framework (Context, Objective, Style, Tone, Audience, Response).
3.  **Few-Shot:** You must provide at least **2 examples** of "Bad Tone" vs. "Good Tone" to teach the AI your brand voice.
4.  **Constraints:** You must explicitly forbid the AI from promising a "Full Refund" (we only offer vouchers).
5.  **Recency:** Ensure the "No Cash Refund" rule is placed where the AI won't forget it.

### Submission Format
Submit your Python code block (like the ones we used in class) along with one sample output generated by your tool.

In [ ]:
# @title (Hidden) Instructor Answer Key
# This is an example of what a "Perfect" submission looks like.

# 1. VARIABLES
C_CONTEXT = "I am the Head of Customer Experience at CloudFly. We are facing a PR crisis regarding a cat incident."
O_OBJECTIVE = "Write a reply to a tweet from an angry user. Acknowledge their pain, apologize, and move to DM."
S_STYLE = "Human, warm, not corporate lawyer-speak."
T_TONE = "Deeply apologetic and humble."
A_AUDIENCE = "An angry social media user (public visibility)."
R_RESPONSE = "A short tweet (max 280 chars)."

# 2. FEW-SHOT
examples = """
BAD RESPONSE: "We apologize for the inconvenience. Please contact support." (Too robotic).
GOOD RESPONSE: "We are absolutely heartbroken to see this. 💔 This is not who we are. Please DM us immediately—we want to make this right."

BAD RESPONSE: "Read our terms of service regarding pets." (Defensive).
GOOD RESPONSE: "I am personally looking into this right now. This is unacceptable. Please check your DMs."
"""

# 3. CONSTRAINTS
constraints = """
- Do NOT promise a cash refund.
- Do NOT argue with the customer.
- Do NOT use the phrase "inconvenience" (it minimizes the issue).
- IMPORTANT: Only offer "Travel Vouchers", never cash.
"""

# 4. ASSEMBLY
tweet_input = "I will never fly CloudFly again! You guys hate animals! #BoycottCloudFly"

final_prompt = f"""
CONTEXT: {C_CONTEXT}
OBJECTIVE: {O_OBJECTIVE}
STYLE: {S_STYLE}
TONE: {T_TONE}
AUDIENCE: {A_AUDIENCE}
RESPONSE: {R_RESPONSE}

EXAMPLES:
{examples}

INCOMING TWEET: "{tweet_input}"

CONSTRAINTS:
{constraints}
"""

print(model.generate_content(final_prompt).text)